# Pixal3D CUDA · Colab T4 · 512 geometry + PBR GLB

Select **Runtime → Change runtime type → T4 GPU**, then run cells in order. Start with a fresh runtime. This notebook installs a separate CUDA runtime, not the MLX repository.

**Status:** one pinned shoe image completed end-to-end on a real T4: 512 geometry, two embedded 1024 PBR textures, 1,083.05 s including uncached downloads, 5.45 GiB peak PyTorch allocated / 5.48 GiB reserved. All 29 tests passed on T4. This is not a general memory guarantee, warm-cache benchmark, or quality-parity claim; visible surface artifacts remain. See [validation evidence](https://github.com/syi0808/pixal3d-cuda/blob/main/docs/validation.md).

You need access to the [official gated DINOv3 checkpoint](https://huggingface.co/facebook/dinov3-vitl16-pretrain-lvd1689m) and a Hugging Face read token. Model weights remain subject to their upstream licenses. Expect roughly 25–30 GB of model downloads and leave at least 40 GiB free disk space.

512 means the **geometry grid**, not a dense 512³ tensor or an image/texture resolution. Refinement/texture still use the official 1024-named checkpoints on a 32³ latent grid. All four flow schedules retain 12 steps.

## 1. Install a T4-compatible environment

PyTorch 2.6.0 / torchvision 0.21.0 / CUDA 12.4 are deliberately paired. No FlashAttention, NATTEN, FlexGEMM, spconv, nvdiffrast, or custom CUDA compilation is required. MoGe-2 is pinned before the MoGe-3 dependency changes.

If PyTorch was already imported, restart the runtime before installation.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

def run_command(command):
    process = subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)


if "torch" in sys.modules:
    raise RuntimeError("Restart the Colab runtime, then run this install cell before importing torch.")

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
REPOSITORY = "https://github.com/syi0808/pixal3d-cuda.git"
REFERENCE = "v0.1.1"
ROOT = Path("/content/pixal3d-cuda")
if not ROOT.exists():
    run_command(["git", "clone", "--depth", "1", "--branch", REFERENCE, REPOSITORY, str(ROOT)])
print("Runtime source:", subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip())
run_command([sys.executable, "-m", "pip", "install", "torch==2.6.0", "torchvision==0.21.0",
                "--index-url", "https://download.pytorch.org/whl/cu124"])
run_command([sys.executable, "-m", "pip", "install", "-e", f"{ROOT}[dev,preprocess,camera]"])
os.chdir(ROOT)

## 2. GPU preflight and operator tests

This tests both FP16 and FP32 attention on the allocated GPU. A different allocated GPU is allowed, but its results are not a T4 measurement.

In [ ]:
import json
import shutil
import torch

assert torch.cuda.is_available(), "Select a GPU runtime first."
print(subprocess.check_output(["nvidia-smi"], text=True))
free_gib = shutil.disk_usage("/content").free / 1024**3
print(f"Free disk: {free_gib:.1f} GiB")
assert free_gib >= 40, "Free at least 40 GiB before downloading weights."
run_command([sys.executable, "-m", "pixalcuda.cli", "doctor"])
run_command([sys.executable, "-m", "pytest", "-q"])

## 3. Hugging Face authentication

Add a Colab secret named **HF_TOKEN** and enable notebook access, or use the interactive login. The token is not placed in shell commands, notebook outputs, Git, or the generated report.

In [ ]:
from huggingface_hub import login, notebook_login

try:
    from google.colab import userdata
    secret = userdata.get("HF_TOKEN")
except Exception:
    secret = None

if secret:
    login(token=secret, add_to_git_credential=False)
    del secret
else:
    notebook_login()

## 4. Download and validate pinned model checkpoints

Only one model is materialized at a time during generation. The validation below checks every expected parameter name and shape using meta models. It is not an inference-quality test.

In [ ]:
run_command([sys.executable, "-m", "pixalcuda.cli", "download"])
run_command([sys.executable, "scripts/verify_checkpoints.py"])

## 5. Upload an image and choose output storage

A single object with a clean background is a good first input. RGBA alpha is respected. Otherwise CPU background removal is used.

Enable Google Drive to preserve checkpoints through runtime loss. Without Drive, download the output archive before disconnecting. On reconnect, reinstall/authenticate/upload the **same original image** and set RESUME=True with the same settings and run name.

In [ ]:
from google.colab import files
from IPython.display import display
from PIL import Image

USE_DRIVE = False
RUN_NAME = "first-t4-512"
RESUME = False
SEED = 42
TEXTURE_SIZE = 1024
CAMERA = "auto"
FOV_DEGREES = 49.13

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive/pixal3d-cuda/runs")
else:
    OUTPUT_ROOT = ROOT / "outputs"

uploaded = files.upload()
assert len(uploaded) == 1, "Upload exactly one image."
filename, payload = next(iter(uploaded.items()))
INPUT = Path("/content/pixal3d-input") / Path(filename).name
INPUT.parent.mkdir(exist_ok=True)
INPUT.write_bytes(payload)
del uploaded, payload
with Image.open(INPUT) as source:
    display(source.copy())
OUTPUT = OUTPUT_ROOT / RUN_NAME
print("Output directory:", OUTPUT)

## 6. Generate 512 geometry + PBR textures

This retains SS → LR shape → coordinate subdivision → HR refinement → guided shape/PBR decode. It does not silently lower resolution, remove geometry tokens, reduce steps, quantize weights, or substitute bilinear features for NAF.

Start with the defaults. If an explicit token/voxel guard triggers, the report explains the stage; increasing that guard may cause CUDA OOM. An interrupted run can resume completed stages using RESUME=True.

In [ ]:
command = [
    sys.executable, "-u", "-m", "pixalcuda.cli", "generate",
    "--image", str(INPUT), "--output-dir", str(OUTPUT),
    "--seed", str(SEED), "--texture-size", str(TEXTURE_SIZE),
    "--target-faces", "100000", "--camera", CAMERA, "--fov-degrees", str(FOV_DEGREES),
    "--attention-queries", "256", "--ffn-rows", "1024",
    "--sparse-rows", "4096", "--naf-queries", "128",
    "--max-tokens", "16384", "--max-voxels", "1500000",
]
if RESUME:
    command.append("--resume")
run_command(command)

## 7. Validate, inspect resources, and preview the actual GLB

The preview reads the exported GLB, not the input image. Its viewer JavaScript is loaded from the model-viewer CDN. If the embedded viewer is unavailable, download the GLB and open it in a glTF/PBR viewer.

In [ ]:
from pixalcuda.export import validate_glb
from IPython.display import HTML
import base64

GLB = OUTPUT / "result.glb"
print(json.dumps(validate_glb(GLB), indent=2))
report = json.loads((OUTPUT / "report.json").read_text())
print("Status:", report["status"])
for stage in report["stages"]:
    allocated = stage.get("peak_allocated_bytes", 0) / 1024**3
    reserved = stage.get("peak_reserved_bytes", 0) / 1024**3
    rss = stage["peak_sampled_rss_bytes"] / 1024**3
    print(f'{stage["name"]:24s} {stage["elapsed_seconds"]:8.1f}s  allocated={allocated:.2f} GiB  reserved={reserved:.2f} GiB  sampled RSS={rss:.2f} GiB')

encoded_glb = base64.b64encode(GLB.read_bytes()).decode("ascii")
display(HTML(
    '<script type="module" src="https://unpkg.com/@google/model-viewer@3.5.0/dist/model-viewer.min.js"></script>'
    '<model-viewer src="data:model/gltf-binary;base64,' + encoded_glb + '" '
    'camera-controls auto-rotate shadow-intensity="1" '
    'style="width:100%;height:540px;background:#20242b"></model-viewer>'
))

## 8. Download results

The archive includes the processed input, intermediate tensors, and report. Treat it as private if your image is private. It contains no model weights or Hugging Face token.

In [ ]:
files.download(str(GLB))
archive = shutil.make_archive("/content/pixal3d-result", "zip", root_dir=OUTPUT)
files.download(archive)